In [0]:
dbutils.notebook.run(
    "/Users/aadityrathore0344@gmail.com/financial-market-intelligence/Set_UP/store_key",
    timeout_seconds=60
)
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Bronze read karo
print("📖 Reading bronze forex data...")
bronze_df = spark.table("financial_market.bronze.forex")
print(f"Bronze records: {bronze_df.count():,}")

# Schema check
print("\nSchema:")
bronze_df.printSchema()

# Quality check
print("\n📊 Initial Data Quality Check:")
print(f"Null date:            {bronze_df.filter(F.col('date').isNull()).count()}")
print(f"Null exchange_rate:   {bronze_df.filter(F.col('exchange_rate').isNull()).count()}")
print(f"Null pair:            {bronze_df.filter(F.col('pair').isNull()).count()}")
print(f"Null base_currency:   {bronze_df.filter(F.col('base_currency').isNull()).count()}")

display(bronze_df.limit(5))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Bronze read karo
print("📖 Reading bronze forex data...")
bronze_df = spark.table("financial_market.bronze.forex")
print(f"Bronze records: {bronze_df.count():,}")

# Schema check
print("\nSchema:")
bronze_df.printSchema()

# Quality check
print("\n📊 Initial Data Quality Check:")
print(f"Null date:            {bronze_df.filter(F.col('date').isNull()).count()}")
print(f"Null exchange_rate:   {bronze_df.filter(F.col('exchange_rate').isNull()).count()}")
print(f"Null pair:            {bronze_df.filter(F.col('pair').isNull()).count()}")
print(f"Null base_currency:   {bronze_df.filter(F.col('base_currency').isNull()).count()}")

display(bronze_df.limit(5))

In [0]:
print("✨ Starting cleaning pipeline...\n")

# Step 1 — null critical values remove karo
initial = bronze_df.count()
cleaned_df = bronze_df.filter(
    F.col("base_currency").isNotNull() &
    F.col("target_currency").isNotNull() &
    F.col("exchange_rate").isNotNull() &
    F.col("date").isNotNull()
)
print(f"✅ After null remove: {cleaned_df.count():,}")

# Step 2 — invalid rates remove karo
cleaned_df = cleaned_df.filter(
    (F.col("exchange_rate") > 0) &
    (F.col("exchange_rate") < 1000000)
)
print(f"✅ After rate validation: {cleaned_df.count():,}")

# Step 3 — date column ensure DateType hai
cleaned_df = cleaned_df \
    .withColumn("date", F.to_date(F.col("date")))

# Step 4 — deduplicate by pair + date
window_spec = Window \
    .partitionBy("base_currency", "target_currency", "date") \
    .orderBy(F.col("fetch_timestamp").desc())

cleaned_df = cleaned_df \
    .withColumn("row_num", F.row_number().over(window_spec)) \
    .filter(F.col("row_num") == 1) \
    .drop("row_num")

print(f"✅ After deduplication: {cleaned_df.count():,}")

# Step 5 — derived columns add karo
cleaned_df = cleaned_df \
    .withColumn("inverse_rate",
                F.round(1 / F.col("exchange_rate"), 6)) \
    .withColumn("inverse_pair",
                F.concat(
                    F.col("target_currency"),
                    F.lit("/"),
                    F.col("base_currency")
                ))

# Step 6 — metadata add karo
cleaned_df = cleaned_df \
    .withColumn("data_quality", F.lit("clean")) \
    .withColumn("processed_at", F.current_timestamp())

print(f"\n✅ Cleaning complete")
print(f"✅ Final row count: {cleaned_df.count():,}")
print(f"✅ Unique pairs: {cleaned_df.select('pair').distinct().count()}")
print(f"✅ Date range: {cleaned_df.agg(F.min('date')).collect()[0][0]} to {cleaned_df.agg(F.max('date')).collect()[0][0]}")

display(cleaned_df.limit(10))

In [0]:
print("💾 Saving to Silver layer...")

cleaned_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("base_currency") \
    .saveAsTable("financial_market.silver.forex")

print(f"✅ Saved to: financial_market.silver.forex")

# verify
df_verify = spark.sql("""
    SELECT
        pair,
        COUNT(*)                      as total_rows,
        MIN(date)                     as earliest_date,
        MAX(date)                     as latest_date,
        ROUND(AVG(exchange_rate), 4)  as avg_rate
    FROM financial_market.silver.forex
    GROUP BY pair
    ORDER BY pair
""")

print(f"\n✅ Silver forex verification:")
print(f"✅ Total rows: {spark.table('financial_market.silver.forex').count():,}")
display(df_verify)